# Exploratory Data Analysis: `diabetes_hypertension_1`

Source file: `diabetes___hypertension_1.csv`

This notebook performs a structured EDA: data loading, structure/dtypes, missing-value analysis, descriptive statistics, distributions, categorical breakdowns, and a correlation heatmap, finishing with a single combined dashboard figure saved as PNG.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=0.8)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

dataset_name = 'diabetes_hypertension_1'


## 1. Load data

In [2]:
csv_path = r"/mnt/user-data/uploads/diabetes___hypertension_1.csv"
try:
    df = pd.read_csv(csv_path, low_memory=False)
except UnicodeDecodeError:
    df = pd.read_csv(csv_path, low_memory=False, encoding='latin1')

print("Shape:", df.shape)
df.head()


Shape: (70692, 18)


,Age,Sex,HighChol,CholCheck,BMI,Smoker,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,GenHlth,MentHlth,PhysHlth,DiffWalk,Stroke,HighBP,Diabetes
0,4.0,1.0,0.0,1.0,26.0,0.0,0.0,1.0,0.0,1.0,0.0,3.0,5.0,30.0,0.0,0.0,1.0,0.0
1,12.0,1.0,1.0,1.0,26.0,1.0,0.0,0.0,1.0,0.0,0.0,3.0,0.0,0.0,0.0,1.0,1.0,0.0
2,13.0,1.0,0.0,1.0,26.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,10.0,0.0,0.0,0.0,0.0
3,11.0,1.0,1.0,1.0,28.0,1.0,0.0,1.0,1.0,1.0,0.0,3.0,0.0,3.0,0.0,0.0,1.0,0.0
4,8.0,0.0,0.0,1.0,29.0,1.0,0.0,1.0,1.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0


## 2. Structure & data types

In [3]:
dtype_counts = df.dtypes.astype(str).value_counts()
print("Column count by dtype:")
print(dtype_counts)
print()
buf_info = []
df.info(buf=None)


Column count by dtype:
float64    18
Name: count, dtype: int64

<class 'pandas.DataFrame'>
RangeIndex: 70692 entries, 0 to 70691
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Age                   70692 non-null  float64
 1   Sex                   70692 non-null  float64
 2   HighChol              70692 non-null  float64
 3   CholCheck             70692 non-null  float64
 4   BMI                   70692 non-null  float64
 5   Smoker                70692 non-null  float64
 6   HeartDiseaseorAttack  70692 non-null  float64
 7   PhysActivity          70692 non-null  float64
 8   Fruits                70692 non-null  float64
 9   Veggies               70692 non-null  float64
 10  HvyAlcoholConsump     70692 non-null  float64
 11  GenHlth               70692 non-null  float64
 12  MentHlth              70692 non-null  float64
 13  PhysHlth              70692 non-null  float64
 14  DiffWalk         

## 3. Missing value analysis

In [4]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(f"Columns with missing values: {len(missing_df)} / {df.shape[1]}")
missing_df.head(30)


Columns with missing values: 0 / 18


,missing_count,missing_pct


## 4. Descriptive statistics (numeric columns)

In [5]:
numeric_cols_all = df.select_dtypes(include=np.number).columns.tolist()
desc = df[numeric_cols_all].describe().T
desc.head(30)


,count,mean,std,min,25%,50%,75%,max
Age,70692.0,8.584055,2.852153,1.0,7.0,9.0,11.0,13.0
Sex,70692.0,0.456997,0.498151,0.0,0.0,0.0,1.0,1.0
HighChol,70692.0,0.525703,0.499342,0.0,0.0,1.0,1.0,1.0
CholCheck,70692.0,0.975259,0.155336,0.0,1.0,1.0,1.0,1.0
BMI,70692.0,29.856985,7.113954,12.0,25.0,29.0,33.0,98.0
Smoker,70692.0,0.475273,0.499392,0.0,0.0,0.0,1.0,1.0
HeartDiseaseorAttack,70692.0,0.147810,0.354914,0.0,0.0,0.0,0.0,1.0
PhysActivity,70692.0,0.703036,0.456924,0.0,0.0,1.0,1.0,1.0
Fruits,70692.0,0.611795,0.487345,0.0,0.0,1.0,1.0,1.0
Veggies,70692.0,0.788774,0.408181,0.0,1.0,1.0,1.0,1.0


## 5. Select representative columns for visualization

Given the potentially large number of columns, we pick the most complete (least missing) numeric columns and the most informative low-cardinality categorical columns for plotting.

In [6]:
id_like = []

# Numeric columns: exclude ID-like, exclude near-constant, rank by completeness then variance
num_candidates = [c for c in numeric_cols_all if c not in id_like]
completeness = df[num_candidates].notna().mean()
nunique = df[num_candidates].nunique()
num_candidates = [c for c in num_candidates if nunique.get(c, 0) > 1]
ranked_numeric = (completeness.loc[num_candidates]).sort_values(ascending=False)
plot_numeric_cols = ranked_numeric.head(12).index.tolist()
print("Numeric columns selected for plotting:")
print(plot_numeric_cols)

# Categorical-like columns: object/string dtype OR numeric with small nunique
obj_cols = df.select_dtypes(exclude=np.number).columns.tolist()
low_card_numeric = [c for c in numeric_cols_all if c not in id_like and 2 <= df[c].nunique() <= 10]
cat_candidates = [c for c in (obj_cols + low_card_numeric) if c not in id_like]
cat_candidates = [c for c in cat_candidates if df[c].nunique() <= 15 and df[c].nunique() >= 2]
plot_cat_cols = cat_candidates[:6]
print("Categorical columns selected for plotting:")
print(plot_cat_cols)


Numeric columns selected for plotting:
['Age', 'Sex', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'GenHlth']


Categorical columns selected for plotting:
['Sex', 'HighChol', 'CholCheck', 'Smoker', 'HeartDiseaseorAttack', 'PhysActivity']


## 6. Correlation heatmap

In [7]:
corr_cols = ranked_numeric.head(15).index.tolist()
plt.figure(figsize=(9, 7))
if len(corr_cols) >= 2:
    corr = df[corr_cols].corr()
    sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, linewidths=0.3, cbar_kws={'shrink': 0.7})
    plt.title(f'Correlation heatmap ({dataset_name})')
    plt.tight_layout()
else:
    plt.text(0.5, 0.5, 'Not enough numeric columns for correlation', ha='center')
plt.show()


## 7. Distributions of key numeric variables

In [8]:
n = len(plot_numeric_cols)
ncols = 4
nrows = int(np.ceil(n / ncols)) if n else 1
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3.5, nrows*2.8))
axes = np.array(axes).reshape(-1)
for i, c in enumerate(plot_numeric_cols):
    ax = axes[i]
    data = df[c].dropna()
    ax.hist(data, bins=30, color='#4C72B0', edgecolor='white')
    ax.set_title(c, fontsize=9)
for j in range(len(plot_numeric_cols), len(axes)):
    axes[j].axis('off')
fig.suptitle(f'Distributions of key numeric variables ({dataset_name})', y=1.02)
plt.tight_layout()
plt.show()


## 8. Categorical variable breakdowns

In [9]:
n = len(plot_cat_cols)
if n:
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*3))
    axes = np.array(axes).reshape(-1)
    for i, c in enumerate(plot_cat_cols):
        ax = axes[i]
        vc = df[c].value_counts(dropna=True).head(10)
        sns.barplot(x=vc.values, y=vc.index.astype(str), ax=ax, color='#55A868')
        ax.set_title(c, fontsize=9)
        ax.set_xlabel('')
    for j in range(len(plot_cat_cols), len(axes)):
        axes[j].axis('off')
    fig.suptitle(f'Categorical variable breakdowns ({dataset_name})', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No suitable low-cardinality categorical columns found.')


## 9. Combined EDA dashboard (single PNG canvas)

All the small plots above are combined into one figure/canvas and exported as a single PNG image for this dataset.

In [10]:
png_path = r"/mnt/user-data/outputs/diabetes_hypertension_1_EDA_dashboard.png"

n_num = len(plot_numeric_cols)
n_cat = len(plot_cat_cols)
hist_rows = int(np.ceil(n_num / 4)) if n_num else 1
cat_rows = int(np.ceil(n_cat / 3)) if n_cat else 0

total_rows = 2 + hist_rows + cat_rows  # row0: missing+corr, then hist_rows, then cat_rows
fig = plt.figure(figsize=(20, 5 + hist_rows*3.2 + cat_rows*3.2))
gs = gridspec.GridSpec(1 + hist_rows + cat_rows, 4, figure=fig, hspace=0.6, wspace=0.4)

# Row 0a: missingness (spans 2 cols)
ax_miss = fig.add_subplot(gs[0, 0:2])
if len(missing_df):
    top_missing = missing_df.head(15)
    sns.barplot(x=top_missing['missing_pct'], y=top_missing.index.astype(str), ax=ax_miss, color='#C44E52')
    ax_miss.set_title('Top missing-value columns (%)', fontsize=10)
else:
    ax_miss.text(0.5, 0.5, 'No missing values', ha='center')
    ax_miss.axis('off')

# Row 0b: correlation heatmap (spans 2 cols)
ax_corr = fig.add_subplot(gs[0, 2:4])
if len(corr_cols) >= 2:
    sns.heatmap(df[corr_cols].corr(), cmap='coolwarm', center=0, ax=ax_corr, cbar_kws={'shrink': 0.6})
    ax_corr.set_title('Correlation heatmap', fontsize=10)
    ax_corr.tick_params(labelsize=6)
else:
    ax_corr.text(0.5, 0.5, 'Not enough numeric cols', ha='center')
    ax_corr.axis('off')

# Histogram grid
for i, c in enumerate(plot_numeric_cols):
    r = 1 + i // 4
    cpos = i % 4
    ax = fig.add_subplot(gs[r, cpos])
    ax.hist(df[c].dropna(), bins=25, color='#4C72B0', edgecolor='white')
    ax.set_title(c, fontsize=8)
    ax.tick_params(labelsize=6)

# Categorical grid
for i, c in enumerate(plot_cat_cols):
    r = 1 + hist_rows + i // 3
    cpos_group = i % 3
    ax = fig.add_subplot(gs[r, cpos_group])
    vc = df[c].value_counts(dropna=True).head(8)
    sns.barplot(x=vc.values, y=vc.index.astype(str), ax=ax, color='#55A868')
    ax.set_title(c, fontsize=8)
    ax.tick_params(labelsize=6)
    ax.set_xlabel('')

fig.suptitle(f'Combined EDA Dashboard — diabetes_hypertension_1', fontsize=16, y=1.01)
fig.savefig(png_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved combined dashboard to:', png_path)


Saved combined dashboard to: /mnt/user-data/outputs/diabetes_hypertension_1_EDA_dashboard.png


## 10. Summary

This notebook covered: dataset shape and dtypes, missingness, descriptive statistics, univariate distributions of the most complete numeric variables, categorical breakdowns of low-cardinality fields, a correlation heatmap, and a single combined dashboard PNG summarizing all of the above.